# AKO: Agentic Kernel Optimization (real measurements)

An LLM writes **Triton** GPU kernels. Each candidate is **checked for correctness** against PyTorch and **timed** on the GPU. The results are fed back, and **Jev** (a System One decision model) picks the next optimization move. A **random-move** run is the ablation, standing in for the old placeholder RL agent.

**Setup (one time):**
1. Runtime → Change runtime type → **T4 GPU**.
2. Click 🔑 **Secrets** in the left sidebar, add `OPENROUTER_API_KEY`, and turn on notebook access. The key never goes in the notebook.
3. Runtime → **Run all**. It takes about 15–30 minutes and roughly $1 of OpenRouter usage with the default settings.

Triton also targets AMD ROCm, so the same harness runs on an MI300X with a ROCm build of PyTorch. Every candidate runs in its **own process**, so a bad kernel (crash, illegal memory access or hang) can't kill the notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")   # from Colab Secrets, never printed
assert OPENROUTER_API_KEY, "Add OPENROUTER_API_KEY in the Colab Secrets panel"

LLM_MODEL = "anthropic/claude-sonnet-4.5"   # any OpenRouter model id
JEV_MODEL = "typesafe/jev-1.13"
TASK_NAMES = ["softmax", "matmul"]
POLICIES = ["jev", "random"]                 # "random" = ablation (the old placeholder agent)
MAX_ITERS = 6
SEED = 0

In [ ]:
%%writefile /content/ako_tasks.py
"""AKO evaluator. Runs ONE candidate kernel in its own process, so a crash or hang can't kill the notebook.
Usage: python ako_tasks.py <task> <candidate.py>  ->  prints 'AKO_RESULT {json}'."""
import importlib.util, json, pathlib, re, sys, traceback
import torch, triton

DEV = "cuda"


def softmax_inputs():
    torch.manual_seed(0)
    return (torch.randn(4096, 2048, device=DEV, dtype=torch.float32),)


def matmul_inputs():
    torch.manual_seed(0)
    return (torch.randn(2048, 2048, device=DEV, dtype=torch.float16),
            torch.randn(2048, 2048, device=DEV, dtype=torch.float16))


TASKS = {
    "softmax": dict(
        spec="Row-wise softmax over the last dimension of a float32 tensor x of shape (4096, 2048). "
             "Return a new float32 tensor of the same shape.",
        inputs=softmax_inputs,
        reference=lambda x: torch.softmax(x, dim=-1),
        tol=dict(atol=1e-5, rtol=1e-4),
        bytes=lambda x: 2 * x.numel() * x.element_size(),  # ideal: read x once, write y once
        flops=None,
        forbidden=r"softmax\(",
    ),
    "matmul": dict(
        spec="Matrix multiply C = A @ B for float16 A (2048x2048) and B (2048x2048). "
             "Accumulate in float32 and return a float16 C of shape (2048, 2048).",
        inputs=matmul_inputs,
        reference=lambda a, b: a @ b,
        tol=dict(atol=1e-1, rtol=1e-2),
        bytes=None,
        flops=lambda a, b: 2 * a.shape[0] * a.shape[1] * b.shape[1],
        # torch.matmul/mm/bmm/einsum, x.matmul(...), or the a @ b operator (decorators like @triton.jit are fine)
        forbidden=r"torch\.(matmul|mm|bmm|einsum)\b|\.matmul\(|[\w\)\]][ \t]*@[ \t]*[\w\(]",
    ),
}


def throughput(t, inputs, ms):
    r = {}
    if t["bytes"]:
        r["gbps"] = t["bytes"](*inputs) / (ms * 1e-3) / 1e9
    if t["flops"]:
        r["tflops"] = t["flops"](*inputs) / (ms * 1e-3) / 1e12
    return r


def baseline(name):
    t = TASKS[name]
    inputs = t["inputs"]()
    ms = triton.testing.do_bench(lambda: t["reference"](*inputs))
    return dict(ms=ms, **throughput(t, inputs, ms))


def short_tb(e, limit=1500):
    return "".join(traceback.format_exception(type(e), e, e.__traceback__))[-limit:]


def evaluate(name, path):
    t = TASKS[name]
    code = pathlib.Path(path).read_text()
    if "triton.jit" not in code:
        return dict(status="rejected", error="No @triton.jit kernel found.")
    if re.search(t["forbidden"], code):
        return dict(status="rejected", error="Calls the PyTorch reference op instead of doing the work in a Triton kernel.")
    inputs = t["inputs"]()
    ref = t["reference"](*inputs)
    try:
        spec = importlib.util.spec_from_file_location(pathlib.Path(path).stem, path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)  # Triton's @jit reads the source, so it must live in a real file
        out = mod.run(*[x.clone() for x in inputs])
        torch.cuda.synchronize()
    except Exception as e:
        return dict(status="error", error=short_tb(e))
    if not isinstance(out, torch.Tensor) or out.shape != ref.shape or out.dtype != ref.dtype:
        got = (tuple(out.shape), str(out.dtype)) if isinstance(out, torch.Tensor) else type(out).__name__
        return dict(status="wrong", error=f"Expected shape/dtype {(tuple(ref.shape), str(ref.dtype))}, got {got}.")
    if not torch.allclose(out, ref, **t["tol"]):
        err = (out.float() - ref.float()).abs().max().item()
        return dict(status="wrong", error=f"Max abs error {err:.3g} exceeds tolerance {t['tol']}.")
    try:
        ms = triton.testing.do_bench(lambda: mod.run(*inputs))
    except Exception as e:
        return dict(status="error", error=short_tb(e))
    return dict(status="ok", ms=ms, **throughput(t, inputs, ms))


if __name__ == "__main__":
    print("AKO_RESULT " + json.dumps(evaluate(sys.argv[1], sys.argv[2])))


In [ ]:
import sys, json, subprocess, pathlib, importlib
sys.path.insert(0, "/content")
import ako_tasks; importlib.reload(ako_tasks)

GPU = torch.cuda.get_device_name(0)
KDIR = pathlib.Path("/content/ako_kernels"); KDIR.mkdir(exist_ok=True)

def run_candidate(task, code, tag):
    path = KDIR / f"{tag}.py"
    path.write_text(code)
    try:
        p = subprocess.run([sys.executable, "/content/ako_tasks.py", task, str(path)],
                           capture_output=True, text=True, timeout=180)
    except subprocess.TimeoutExpired:
        return {"status": "error", "error": "Timed out after 180 s (kernel hang or very slow compile)."}
    for line in reversed(p.stdout.splitlines()):
        if line.startswith("AKO_RESULT "):
            return json.loads(line[len("AKO_RESULT "):])
    return {"status": "error", "error": ((p.stderr or p.stdout)[-1500:] or f"process exited {p.returncode}")}

BASELINE = {name: ako_tasks.baseline(name) for name in TASK_NAMES}
BASELINE

## Agent: LLM generator + Jev planner
- **Last attempt failed** → `fix_correctness` (a code rule, no model call).
- **Last attempt worked** → the policy picks the next move. `jev` asks Jev a *choice* question over the moves (including `stop`). `random` picks a move at random and never stops early.

In [ ]:
import re, random, time, requests

HEADERS = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}

MOVES = {
    "tune_launch_config": "Change block sizes, num_warps or num_stages, or add @triton.autotune over a few configs.",
    "improve_memory_access": "Improve memory access: coalesced loads, fewer global reads/writes, keep data in registers/shared memory.",
    "restructure_algorithm": "Change the algorithm: tiling strategy, fusing passes, reduction order, or grouped ordering for cache reuse.",
}
FIX = "The previous attempt failed (see the error). Fix it while keeping the design."

SYSTEM = (
    "You write GPU kernels in OpenAI Triton (Python). Reply with exactly ONE ```python block holding a complete module: "
    "imports (torch, triton, triton.language as tl), one or more @triton.jit kernels, and a function `run(*inputs)` that "
    "allocates the output tensor, launches the kernel(s) and returns the output. The work must happen inside your Triton "
    "kernels: do not call the PyTorch op being implemented (e.g. torch.softmax, torch.matmul or the @ operator) in run(). "
    "Use only portable Triton so it also runs on AMD ROCm."
)

def chat(messages):
    r = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=HEADERS, timeout=300,
                      json={"model": LLM_MODEL, "messages": messages, "temperature": 0.2})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def extract_code(text):
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    return max(blocks, key=len) if blocks else text

def summary_line(r):
    s = f"iter {r['iter']} [{r['move']}] {r['status']}"
    if r["status"] == "ok":
        s += f": {r['ms']:.4f} ms, {r['speedup']:.2f}x vs PyTorch"
    return s

def build_prompt(task, log, best, last, move_text):
    t = ako_tasks.TASKS[task]
    parts = [f"Task: {t['spec']}", f"GPU: {GPU}. PyTorch baseline: {BASELINE[task]['ms']:.4f} ms."]
    if log:
        parts.append("Attempts so far:\n" + "\n".join(summary_line(r) for r in log))
    if last and last["status"] != "ok":
        parts.append(f"Previous attempt (failed):\n```python\n{last['code']}\n```\nError:\n{last['error']}")
    elif best:
        parts.append(f"Best kernel so far ({best['ms']:.4f} ms):\n```python\n{best['code']}\n```")
    parts.append(f"Next step: {move_text}" if move_text else "Write a correct, reasonably fast first version.")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(parts)}]

JEV_QUESTIONS = {
    "next_move": {
        "type": "choice",
        "instructions": "Given the optimization `history` of the GPU kernel for `task` (latency and speedup vs the PyTorch "
                        "baseline on `gpu`), which change is most likely to make the best kernel faster next?",
        "criteria": {**MOVES, "stop": "Further gains are unlikely: the best kernel already matches or beats the baseline "
                                      "and recent attempts stopped improving."},
    }
}

def ask_jev(task, log):
    state = {"task": ako_tasks.TASKS[task]["spec"], "gpu": GPU, "baseline_ms": round(BASELINE[task]["ms"], 4),
             "history": [summary_line(r) for r in log]}
    r = requests.post("https://openrouter.ai/api/alpha/decisions", headers=HEADERS, timeout=30,
                      json={"model": JEV_MODEL, "state": state, "questions": JEV_QUESTIONS})
    r.raise_for_status()
    a = r.json()["answers"]["next_move"]
    return a["choice"], a["probabilities"]

STOP_CONF, MIN_ITERS = 0.6, 3

def optimize(task, policy, max_iters=MAX_ITERS):
    rng = random.Random(SEED)
    log, best, last, stopped = [], None, None, None
    for i in range(1, max_iters + 1):
        conf = None
        if i == 1:
            move, move_text = "initial", None
        elif last["status"] != "ok":
            move, move_text = "fix_correctness", FIX
        elif policy == "jev":
            try:
                move, probs = ask_jev(task, log)
                conf = probs[move]
            except Exception as e:
                print(f"  Jev unavailable ({type(e).__name__}), random move this step")
                move = rng.choice(list(MOVES))
            if move == "stop":
                if conf >= STOP_CONF and i > MIN_ITERS:
                    stopped = i
                    print(f"  {task}/{policy}: Jev says stop (p={conf:.2f}) before iter {i}")
                    break
                move = max(MOVES, key=lambda m: probs.get(m, 0))  # unsure stop: take Jev's best real move
                conf = probs.get(move)
            move_text = MOVES[move]
        else:
            move = rng.choice(list(MOVES)); move_text = MOVES[move]

        t0 = time.time()
        try:
            code = extract_code(chat(build_prompt(task, log, best, last, move_text)))
        except Exception as e:
            print(f"  LLM call failed ({e}); skipping iteration"); continue
        gen_s = time.time() - t0
        res = run_candidate(task, code, f"{task}_{policy}_{i}")
        rec = dict(task=task, policy=policy, iter=i, move=move, jev_conf=conf, gen_s=round(gen_s, 1),
                   status=res["status"], ms=res.get("ms"), gbps=res.get("gbps"), tflops=res.get("tflops"),
                   error=res.get("error"), code=code, speedup=None)
        if rec["status"] == "ok":
            rec["speedup"] = BASELINE[task]["ms"] / rec["ms"]
            if best is None or rec["ms"] < best["ms"]:
                best = rec
        log.append(rec); last = rec
        print("  " + summary_line(rec) + ("" if rec["status"] == "ok" else f" | {rec['error'].splitlines()[-1][:120]}"))
    return log, best, stopped

In [ ]:
records, bests, stops = [], {}, {}
for task in TASK_NAMES:
    for policy in POLICIES:
        print(f"=== {task} / {policy} ===")
        log, best, stopped = optimize(task, policy)
        records += log; bests[(task, policy)] = best; stops[(task, policy)] = stopped
        if best:
            (KDIR / f"BEST_{task}_{policy}.py").write_text(best["code"])

In [ ]:
import pandas as pd
df = pd.DataFrame(records)
df.drop(columns=["code"]).to_csv("/content/ako_results.csv", index=False)

rows = []
for (task, policy), g in df.groupby(["task", "policy"], sort=False):
    ok = g[g.status == "ok"]
    b = ok.loc[ok.ms.idxmin()] if len(ok) else None
    rows.append({
        "task": task, "policy": policy, "attempts": len(g),
        "valid_%": round(100 * len(ok) / len(g)),
        "baseline_ms": round(BASELINE[task]["ms"], 4),
        "best_ms": None if b is None else round(b.ms, 4),
        "speedup_vs_torch": None if b is None else round(b.speedup, 2),
        "iter_of_best": None if b is None else int(b["iter"]),
        "best_GB/s": None if b is None or pd.isna(b.gbps) else round(b.gbps),
        "best_TFLOPS": None if b is None or pd.isna(b.tflops) else round(b.tflops, 1),
        "stopped_early_at": stops[(task, policy)],
    })
summary = pd.DataFrame(rows)
summary.to_csv("/content/ako_summary.csv", index=False)
summary

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(TASK_NAMES), figsize=(6 * len(TASK_NAMES), 4), squeeze=False)
for ax, task in zip(axes[0], TASK_NAMES):
    for policy in POLICIES:
        g = df[(df.task == task) & (df.policy == policy)].sort_values("iter")
        best_so_far = g.speedup.fillna(0).cummax().replace(0, float("nan"))
        ax.plot(g["iter"], best_so_far, marker="o", label=policy)
    ax.axhline(1.0, ls="--", c="gray", lw=1, label="PyTorch")
    ax.set(title=f"{task} on {GPU}", xlabel="iteration", ylabel="best speedup vs PyTorch")
    ax.legend()
plt.tight_layout(); plt.savefig("/content/ako_speedup.png", dpi=150); plt.show()

In [ ]:
# Download everything: CSVs, chart, and every generated kernel (BEST_* = winners)
!cd /content && zip -qr ako_results.zip ako_results.csv ako_summary.csv ako_speedup.png ako_kernels
from google.colab import files
files.download("/content/ako_results.zip")

## Reading the results honestly
- **speedup_vs_torch**: above 1 means the LLM's kernel beat PyTorch. For matmul, PyTorch calls cuBLAS (heavily hand-tuned), so a value **near 1 is already good**.
- **valid_%**: how often the LLM's kernels compiled *and* gave correct output. This is the main argument for a correctness gate.
- **jev vs random**: did Jev's move choice reach a better kernel, or reach it sooner? With one run per arm this is **suggestive, not proof**. Say so in the interview.
- **GB/s**: compare it to the T4's ~320 GB/s peak memory bandwidth. Softmax is memory-bound, so this tells you how close it is to the roofline.